# Desarrollo de un Sistema de Generación Aumentada por Recuperación (RAG) para la Consulta y Auditoría de Normativas Internacionales de Inteligencia Artificial.
## Adquisición, Limpieza y Segmentación Documental

**Autor:** Eduard David Apolo Gallardo  
**Máster en Deep Learning — UPM**

---
## Importaciones

In [1]:
import os
import re
import time
import json
import hashlib
import statistics
from pathlib import Path
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredHTMLLoader
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Resolución de la ruta estática local
DATA_DIR = Path(r"E:\Proyectos_IA\Deep_Learning\TFM\Data")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"[ERROR] El directorio base no existe: {DATA_DIR}")

DetectorFactory.seed = 42

print(f"Entorno configurado. Directorio de ingesta: {DATA_DIR.resolve()}")

Entorno configurado. Directorio de ingesta: E:\Proyectos_IA\Deep_Learning\TFM\Data


---
## Ingesta
Se implementa un mapeo de extensiones de archivo para la extracción del contenido independientemente de su distribución en los subdirectorios.

In [2]:
LOADER_MAPPING = {
    ".pdf": PyPDFLoader,
    ".docx": UnstructuredWordDocumentLoader,
    ".doc": UnstructuredWordDocumentLoader,
    ".txt": TextLoader,
    ".html": UnstructuredHTMLLoader,
    ".htm": UnstructuredHTMLLoader
}

documentos_crudos = []
archivos_ignorados = []

print("Iniciando escaneo recursivo del directorio de datos...")

# Recorrido recursivo del árbol de directorios
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        file_path = Path(root) / file
        extension = file_path.suffix.lower()
        
        if extension in LOADER_MAPPING:
            try:
                # Instanciación dinámica del loader correspondiente
                loader_class = LOADER_MAPPING[extension]
                loader = loader_class(str(file_path))
                docs_extraidos = loader.load()
                
                categoria_origen = Path(root).name
                for doc in docs_extraidos:
                    doc.metadata["categoria_fuente"] = categoria_origen
                    doc.metadata["archivo_origen"] = file
                    
                documentos_crudos.extend(docs_extraidos)
            except Exception as e:
                print(f"[ERROR] Fallo al procesar {file_path.name}: {str(e)}")
        else:
            archivos_ignorados.append(file_path.name)

print(f"\nIngesta completada. Total de documentos extraídos: {len(documentos_crudos)}")
if archivos_ignorados:
    print(f"Se ignoraron {len(archivos_ignorados)} archivos por extensión no soportada.")

Iniciando escaneo recursivo del directorio de datos...

Ingesta completada. Total de documentos extraídos: 1548
Se ignoraron 34 archivos por extensión no soportada.


---
## Limpieza
Se aplican expresiones regulares para normalizar el texto y suprimir artefactos tipográficos. Se integra la capa de privacidad para anonimizar datos identificativos.

In [4]:
def limpiar_y_anonimizar(texto: str) -> str:
    """
    Normaliza el texto y enmascara Información de Identificación Personal (PII).
    """
    if not isinstance(texto, str):
        return ""
        
    # Enmascaramiento de correos electrónicos
    texto = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[CORREO_OCULTO]', texto)
    # Enmascaramiento de teléfonos
    texto = re.sub(r'(\+34|0034)?[ -]*(6|7|8|9)[ -]*([0-9][ -]*){8}', '[TELÉFONO_OCULTO]', texto)
    # Limpieza de saltos de línea redundantes
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    # Normalización de espacios múltiples
    texto = re.sub(r' {2,}', ' ', texto)
    
    return texto.strip()

# Aplicación sobre el corpus cargado
documentos_limpios = []
for doc in documentos_crudos:
    texto_procesado = limpiar_y_anonimizar(doc.page_content)
    if len(texto_procesado) > 50: # Descarte de páginas residuales o vacías
        # Se genera un hash único para el contenido procesado, útil para trazabilidad y detección de duplicados
        doc.metadata["hash_sha256"] = hashlib.sha256(texto_procesado.encode("utf-8")).hexdigest()[:16]
        documentos_limpios.append(Document(page_content=texto_procesado, metadata=doc.metadata))

print(f"Limpieza finalizada. Documentos retenidos tras el filtrado de ruido: {len(documentos_limpios)}")

Limpieza finalizada. Documentos retenidos tras el filtrado de ruido: 1460


---
## Chunking
Se particiona el corpus en fragmentos para la vectorización. Se establecen límites estrictos que previenen el desbordamiento de la ventana de contexto de los embeddings.

In [5]:
# Definición de separadores jerárquicos para textos legales y técnicos
SEPARADORES = ["\n\nArtículo ", "\n\nCAPÍTULO ", "\n\n", "\n", ". ", " "]

segmentador = RecursiveCharacterTextSplitter(
    chunk_size=1200,    # Límite adaptado a la densidad terminológica
    chunk_overlap=150,  # Solapamiento para preservación del contexto
    separators=SEPARADORES,
    length_function=len
)

fragmentos_base = segmentador.split_documents(documentos_limpios)
print(f"Fragmentos generados tras segmentación: {len(fragmentos_base)}")

Fragmentos generados tras segmentación: 5209


In [6]:
PATRON_ARTICULO = re.compile(r"Artículo\s+(\d+)\b", re.IGNORECASE)
PATRON_CAPITULO = re.compile(r"CAPÍTULO\s+([IVXLCDM]+|\d+)\b", re.IGNORECASE)

for idx, frag in enumerate(fragmentos_base):
    contenido = frag.page_content
    
    frag.metadata["chunk_id"] = idx
    frag.metadata["longitud_chars"] = len(contenido)
    frag.metadata["hash_sha256"] = hashlib.sha256(contenido.encode("utf-8")).hexdigest()[:16]
    
    coincidencias_art = PATRON_ARTICULO.findall(contenido)
    frag.metadata["articulos"] = list(set(coincidencias_art)) if coincidencias_art else []
    
    coincidencias_cap = PATRON_CAPITULO.findall(contenido)
    frag.metadata["capitulo"] = coincidencias_cap[0] if coincidencias_cap else None

print("Metadatos de trazabilidad inyectados en todos los fragmentos.")

Metadatos de trazabilidad inyectados en todos los fragmentos.


---
## Filtrado de Ruido y Fragmentos Cortos

In [7]:
LONGITUD_MINIMA = 100

n_antes = len(fragmentos_base)
fragmentos_finales = [f for f in fragmentos_base if f.metadata["longitud_chars"] >= LONGITUD_MINIMA]
n_despues = len(fragmentos_finales)

print(f"Fragmentos descartados por falta de contexto (< {LONGITUD_MINIMA} chars): {n_antes - n_despues}")
print(f"Total de fragmentos aptos para vectorización: {n_despues}")

Fragmentos descartados por falta de contexto (< 100 chars): 17
Total de fragmentos aptos para vectorización: 5192


---
## Validación de Calidad del Corpus
La desviación estándar de las longitudes indica la homogeneidad espacial de los vectores. La detección de duplicados por hash previene que la base vectorial asigne pesos redundantes a normativas repetidas durante la inferencia por similitud del coseno.

In [8]:
longitudes = [f.metadata["longitud_chars"] for f in fragmentos_finales]
hashes = set(f.metadata["hash_sha256"] for f in fragmentos_finales)

print("--- AUDITORÍA ESTADÍSTICA DEL CORPUS ---")
print(f"Mínimo de caracteres: {min(longitudes)}")
print(f"Máximo de caracteres: {max(longitudes)}")
print(f"Media de caracteres:  {statistics.mean(longitudes):.1f}")
print(f"Desviación estándar:  {statistics.stdev(longitudes):.1f}")

duplicados = len(fragmentos_finales) - len(hashes)
print(f"\ncolisiones de hash: {duplicados}")

--- AUDITORÍA ESTADÍSTICA DEL CORPUS ---
Mínimo de caracteres: 102
Máximo de caracteres: 1200
Media de caracteres:  985.0
Desviación estándar:  281.4

colisiones de hash: 1


---
## Limpieza de metadatos

In [9]:
ARTICULO_MIN_AI_ACT = 1
ARTICULO_MAX_AI_ACT = 113

corpus = fragmentos_finales  
RUTA_CORPUS_CORREGIDO = DATA_DIR / "corpus_tfm_procesado.json"

n_articulos_filtrados  = 0
n_rutas_normalizadas   = 0

for fragmento in corpus:
    meta = fragmento.metadata

    # Filtrado de artículos por rango válido AI Act
    articulos_raw = meta.get("articulos", [])

    articulos_ai_act  = []
    articulos_externos = []

    for art in articulos_raw:
        try:
            num = int(art)
            if ARTICULO_MIN_AI_ACT <= num <= ARTICULO_MAX_AI_ACT:
                articulos_ai_act.append(str(num))
            else:
                articulos_externos.append(str(num))
                n_articulos_filtrados += 1
        except ValueError:
            articulos_externos.append(art)

    # Se reemplaza los campos
    meta["articulos"]          = articulos_ai_act   # Solo artículos AI Act válidos
    meta["articulos_externos"] = articulos_externos  # Referencias cruzadas a otros reglamentos

    # Se actualiza primer_articulo para facilitar consultas posteriores
    if "primer_articulo" in meta:
        meta["primer_articulo"] = articulos_ai_act[0] if articulos_ai_act else None

    # Normalización de rutas
    source_raw = meta.get("source", "")

    es_ruta_absoluta = (
        bool(re.match(r"^[A-Za-z]:\\", source_raw))  # Windows: E:\
        or source_raw.startswith("/")                  # Unix: /home/...
    )

    if es_ruta_absoluta:
        categoria = meta.get("categoria_fuente", "sin_categoria")
        archivo   = meta.get("archivo_origen", Path(source_raw).name)
        # Formato normalizado:
        meta["source"] = f"{categoria}/{archivo}"
        n_rutas_normalizadas += 1

print("\n--- CORRECCIONES APLICADAS ---")
print(f"  Artículos espurios filtrados (fuera de rango 1-113) : {n_articulos_filtrados}")
print(f"  Rutas normalizadas                                  : {n_rutas_normalizadas}")


--- CORRECCIONES APLICADAS ---
  Artículos espurios filtrados (fuera de rango 1-113) : 1
  Rutas normalizadas                                  : 5192


---
## Validación 

In [10]:
articulos_residuales = []
for fragmento in corpus:
    for art in fragmento.metadata.get("articulos", []):
        try:
            num = int(art)
            if num < ARTICULO_MIN_AI_ACT or num > ARTICULO_MAX_AI_ACT:
                articulos_residuales.append((fragmento.metadata["chunk_id"], art))
        except ValueError:
            pass

print("=" * 55)
print("VALIDACIÓN POST-CORRECCIÓN")
print("=" * 55)

print("\nArtículos fuera de rango residuales:")
if not articulos_residuales:
    print("    [OK] Ninguno. El campo 'articulos' contiene únicamente")
    print("         referencias válidas al AI Act (rango 1-113).")
else:
    print(f"    [AVISO] Se detectaron {len(articulos_residuales)} residuales:")
    for chunk_id, art in articulos_residuales:
        print(f"      Chunk {chunk_id}: artículo {art}")

rutas_absolutas_residuales = [
    (fragmento.metadata["chunk_id"], fragmento.metadata["source"])
    for fragmento in corpus
    if re.match(r"^[A-Za-z]:\\", fragmento.metadata.get("source", ""))
    or fragmento.metadata.get("source", "").startswith("/")
]

print("\nRutas absolutas residuales:")
if not rutas_absolutas_residuales:
    print("    [OK] Ninguna. Todas las rutas están normalizadas.")
else:
    print(f"    [AVISO] {len(rutas_absolutas_residuales)} rutas sin normalizar.")
    for chunk_id, src in rutas_absolutas_residuales[:3]:
        print(f"      Chunk {chunk_id}: {src}")

print("\nMuestra de metadatos corregidos (fragmento 0):")
meta_muestra = corpus[0].metadata
campos_relevantes = [
    "chunk_id", "source", "categoria_fuente",
    "articulos", "articulos_externos", "primer_articulo"
]
for campo in campos_relevantes:
    print(f"    {campo:25s}: {meta_muestra.get(campo, 'N/A')}")

print("\nEjemplo de fragmento con artículos AI Act corregidos:")
for fragmento in corpus:
    if fragmento.metadata.get("articulos"):
        m = fragmento.metadata
        print(f"    chunk_id          : {m['chunk_id']}")
        print(f"    source            : {m['source']}")
        print(f"    articulos         : {m['articulos']}")
        print(f"    articulos_externos: {m.get('articulos_externos', [])}")
        print(f"    primer_articulo   : {m.get('primer_articulo', 'N/A')}")
        print(f"    contenido (200c)  : {fragmento.page_content[:200]}")
        break

VALIDACIÓN POST-CORRECCIÓN

Artículos fuera de rango residuales:
    [OK] Ninguno. El campo 'articulos' contiene únicamente
         referencias válidas al AI Act (rango 1-113).

Rutas absolutas residuales:
    [OK] Ninguna. Todas las rutas están normalizadas.

Muestra de metadatos corregidos (fragmento 0):
    chunk_id                 : 0
    source                   : AEPD EIPD/gestion-riesgo-y-evaluacion-impacto-en-tratamientos-datos-personales.pdf
    categoria_fuente         : AEPD EIPD
    articulos                : []
    articulos_externos       : []
    primer_articulo          : N/A

Ejemplo de fragmento con artículos AI Act corregidos:
    chunk_id          : 1
    source            : AEPD EIPD/gestion-riesgo-y-evaluacion-impacto-en-tratamientos-datos-personales.pdf
    articulos         : ['36']
    articulos_externos: []
    primer_articulo   : N/A
    contenido (200c)  : Página: 2 de 160 
 
 
 
RESUMEN EJECUTIVO 
 
El presente documento es una guía para la gestión de ries

In [11]:
def detectar_idioma(texto: str) -> str:
    """
    Detecta el idioma de un fragmento de texto.
    Se usan los primeros 200 caracteres para priorizar rendimiento.
    """
    try:
        return detect(texto[:200])
    except LangDetectException:
        return "unknown"

print("Etiquetando idioma de los fragmentos...")
t0 = time.time()

contador_idiomas = {}

# 'corpus' es la variable que contiene los fragmentos_finales tras la limpieza de metadatos
for fragmento in corpus:
    idioma = detectar_idioma(fragmento.page_content)
    fragmento.metadata["idioma"] = idioma
    contador_idiomas[idioma] = contador_idiomas.get(idioma, 0) + 1

print(f"Etiquetado completado en {time.time()-t0:.1f}s")
print("\n--- Distribución de idiomas en el corpus ---")
for idioma, count in sorted(contador_idiomas.items(), key=lambda x: -x[1]):
    pct = count / len(corpus) * 100
    print(f"  {idioma:10s}: {count:5d} fragmentos ({pct:.1f}%)")

Etiquetando idioma de los fragmentos...
Etiquetado completado en 12.6s

--- Distribución de idiomas en el corpus ---
  es        :  3243 fragmentos (62.5%)
  en        :  1928 fragmentos (37.1%)
  de        :     7 fragmentos (0.1%)
  pt        :     3 fragmentos (0.1%)
  so        :     2 fragmentos (0.0%)
  cy        :     2 fragmentos (0.0%)
  nl        :     1 fragmentos (0.0%)
  unknown   :     1 fragmentos (0.0%)
  hr        :     1 fragmentos (0.0%)
  it        :     1 fragmentos (0.0%)
  sv        :     1 fragmentos (0.0%)
  fr        :     1 fragmentos (0.0%)
  ca        :     1 fragmentos (0.0%)


---
## Serialización del Corpus

In [12]:
RUTA_SALIDA = DATA_DIR / "corpus_tfm_procesado.json"

corpus_serializable = [
    {"contenido": frag.page_content, "metadatos": frag.metadata}
    for frag in fragmentos_finales
]

with open(RUTA_SALIDA, "w", encoding="utf-8") as f:
    json.dump(corpus_serializable, f, ensure_ascii=False, indent=2)

tamanio_mb = RUTA_SALIDA.stat().st_size / (1024 * 1024)
print(f"Corpus persistido exitosamente en: {RUTA_SALIDA}")
print(f"Tamaño de almacenamiento: {tamanio_mb:.2f} MB")

Corpus persistido exitosamente en: E:\Proyectos_IA\Deep_Learning\TFM\Data\corpus_tfm_procesado.json
Tamaño de almacenamiento: 9.82 MB


In [13]:
print("=" * 60)
print("RESUMEN")
print("=" * 60)
print(f"- Fragmentos consolidados : {len(fragmentos_finales)}")
print(f"- Almacenamiento local    : {RUTA_SALIDA.name}")
print("- Estado                  : Listo")
print("=" * 60)

RESUMEN
- Fragmentos consolidados : 5192
- Almacenamiento local    : corpus_tfm_procesado.json
- Estado                  : Listo
